# 02 — Benchmark các Embedding Model cho School Regulation RAG

Notebook chạy trên **Google Colab A100** và sử dụng dữ liệu đầu ra từ
`01_preprocessing_colab_v2.ipynb`.

## Mục tiêu

So sánh zero-shot các Dense Retriever:

- `Qwen/Qwen3-Embedding-4B`
- `BAAI/bge-m3`
- `intfloat/multilingual-e5-base`
- `bkai-foundation-models/vietnamese-bi-encoder`

Các chỉ số:

- Recall@1, Recall@3, Recall@5, Recall@10, Recall@20
- MRR@10
- nDCG@10
- MAP@10
- thời gian encode corpus/query
- throughput
- peak GPU VRAM
- kích thước embedding

## Nguyên tắc đánh giá

1. Chọn model bằng `val` hoặc `strict_val`.
2. Không dùng `test` để quyết định model/hyperparameter.
3. Sau khi chốt model, mới chạy một lần trên `test` và `strict_test`.
4. Mỗi model dùng đúng input format của model đó.

In [ ]:
# Cài thư viện.
# FlashAttention là tùy chọn; SentenceTransformer vẫn chạy nếu không cài được.
!pip -q install -U \
    transformers \
    sentence-transformers \
    accelerate \
    pyarrow \
    pandas \
    numpy \
    underthesea \
    psutil

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 14.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# 1. IMPORTS AND REPRODUCIBILITY
# ============================================================

from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Iterable
import gc
import json
import math
import os
import random
import time

import numpy as np
import pandas as pd
import psutil
import torch

from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

PyTorch: 2.11.0+cpu
CUDA available: False


In [3]:
# ============================================================
# 2. CONFIG
# ============================================================

PROCESSED_DIR = Path(
    "/content/drive/MyDrive/UIT_Legal_System/Dataset/processed"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/UIT_Legal_System/Dataset/artifacts/embedding_benchmark"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EMBEDDING_CACHE_DIR = OUTPUT_DIR / "embedding_cache"
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Chỉ dùng validation để lựa chọn model.
# Có thể đổi thành ["strict_val"] để đo tổng quát hóa nghiêm ngặt.
EVAL_SPLITS = ["val", "strict_val"]

# Chỉ bật sau khi đã chốt model bằng validation.
RUN_FINAL_TEST = False
FINAL_TEST_SPLITS = ["test", "strict_test"]

# Chọn model để đánh giá final test sau khi benchmark validation.
SELECTED_MODEL_KEY = "qwen3_embedding_4b"

# So sánh hai cách biểu diễn passage:
# - content: chỉ context
# - embedding_text: document + article + context
CORPUS_FIELDS = ["content", "embedding_text"]

TOP_K_VALUES = [1, 3, 5, 10, 20]
MAX_RANK = max(TOP_K_VALUES)

# Độ dài dùng chung để benchmark công bằng theo dữ liệu thực tế.
# Có thể chỉnh sau khi xem token_length_summary.csv.
MAX_QUERY_LENGTH = 128
MAX_PASSAGE_LENGTH = 512

# Batch riêng cho từng model, có thể chỉnh theo A100 40GB/80GB.
DEFAULT_QUERY_BATCH_SIZE = 32
DEFAULT_PASSAGE_BATCH_SIZE = 16

# Cache embedding lên Google Drive để resume khi Colab bị ngắt.
USE_EMBEDDING_CACHE = True
FORCE_RECOMPUTE = False

# Dùng BF16 trên A100.
TORCH_DTYPE = torch.bfloat16

QWEN_TASK_INSTRUCTION = (
    "Given a Vietnamese question about school regulations, "
    "retrieve the most relevant regulation passage that provides "
    "the factual basis needed to answer the question."
)

print("Processed data:", PROCESSED_DIR)
print("Benchmark output:", OUTPUT_DIR)

Processed data: /content/drive/MyDrive/UIT_Legal_System/Dataset/processed
Benchmark output: /content/drive/MyDrive/UIT_Legal_System/Dataset/artifacts/embedding_benchmark


## Format đầu vào theo từng model

- **Qwen3 Embedding:** query có instruction; document không thêm instruction.
- **multilingual-E5:** query dùng `query:`, passage dùng `passage:`.
- **BGE-M3:** không cần instruction cho query.
- **BKAI:** văn bản phải được word segmentation trước khi đưa vào PhoBERT.

In [4]:
# ============================================================
# 3. MODEL SPECIFICATIONS
# ============================================================

@dataclass(frozen=True)
class ModelSpec:
    key: str
    model_name: str
    query_batch_size: int
    passage_batch_size: int
    max_query_length: int
    max_passage_length: int
    query_format: str
    passage_format: str
    word_segment_vietnamese: bool = False
    use_flash_attention: bool = False
    truncate_dim: int | None = None


MODEL_SPECS = {

    "bge_m3": ModelSpec(
        key="bge_m3",
        model_name="BAAI/bge-m3",
        query_batch_size=DEFAULT_QUERY_BATCH_SIZE,
        passage_batch_size=DEFAULT_PASSAGE_BATCH_SIZE,
        max_query_length=MAX_QUERY_LENGTH,
        max_passage_length=MAX_PASSAGE_LENGTH,
        query_format="raw",
        passage_format="raw",
    ),
    "multilingual_e5_base": ModelSpec(
        key="multilingual_e5_base",
        model_name="intfloat/multilingual-e5-base",
        query_batch_size=64,
        passage_batch_size=32,
        max_query_length=MAX_QUERY_LENGTH,
        max_passage_length=min(MAX_PASSAGE_LENGTH, 512),
        query_format="e5_query",
        passage_format="e5_passage",
    ),
    "bkai_vietnamese_bi_encoder": ModelSpec(
        key="bkai_vietnamese_bi_encoder",
        model_name="bkai-foundation-models/vietnamese-bi-encoder",
        query_batch_size=64,
        passage_batch_size=32,
        max_query_length=min(MAX_QUERY_LENGTH, 256),
        max_passage_length=min(MAX_PASSAGE_LENGTH, 256),
        query_format="raw",
        passage_format="raw",
        word_segment_vietnamese=True,
    ),
}

pd.DataFrame([asdict(spec) for spec in MODEL_SPECS.values()])

,key,model_name,query_batch_size,passage_batch_size,max_query_length,max_passage_length,query_format,passage_format,word_segment_vietnamese,use_flash_attention,truncate_dim
0,bge_m3,BAAI/bge-m3,32,16,128,512,raw,raw,False,False,None
1,multilingual_e5_base,intfloat/multilingual-e5-base,64,32,128,512,e5_query,e5_passage,False,False,None
2,bkai_vietnamese_bi_encoder,bkai-foundation-models/vietnamese-bi-encoder,64,32,128,256,raw,raw,True,False,None


In [5]:
# ============================================================
# 4. LOAD PREPROCESSED DATA
# ============================================================

required_files = [
    PROCESSED_DIR / "corpus.parquet",
    PROCESSED_DIR / "val_retriever.parquet",
    PROCESSED_DIR / "val_qrels.json",
]

missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "Thiếu dữ liệu preprocessing:\n- "
        + "\n- ".join(missing_files)
        + "\nHãy chạy notebook 01 trước."
    )

corpus_df = pd.read_parquet(PROCESSED_DIR / "corpus.parquet")

expected_corpus_columns = {
    "passage_id",
    "content",
    "embedding_text",
    "document",
    "article",
}

missing_corpus_columns = expected_corpus_columns - set(corpus_df.columns)

if missing_corpus_columns:
    raise ValueError(
        f"corpus.parquet thiếu cột: {sorted(missing_corpus_columns)}"
    )

passage_ids = corpus_df["passage_id"].astype(str).tolist()
passage_id_to_index = {
    passage_id: index
    for index, passage_id in enumerate(passage_ids)
}

print("Corpus passages:", f"{len(corpus_df):,}")
display(corpus_df.head(3))

Corpus passages: 1,045


,passage_id,title,document,article,content,embedding_text,embedding_text_tokens,qa_count,content_hash
0,passage_7a4034bfcca9a2af6225,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG - Điều ...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂNG,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,Điều 9. Tuyển bổ sung và loại ra khỏi chương t...,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH TÀI NĂN...,1113,65,content_7fce8ab9b5ca93015231
1,passage_a9f2bb6e5bcaf05aa8c3,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ ĐẠI HỌC ...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,Điều 4. Kiểm tra xếp lớp đầu khóa cho sinh viê...,Văn bản: QUY ĐỊNH ĐÀO TẠO NGOẠI NGỮ ĐỐI VỚI HỆ...,768,61,content_8ce05a028a6c8e694978
2,passage_4a0ee8a7dcf2c8b4a016,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO -...,QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯỢNG CAO,Điều 5. Chương trình đào tạo,Điều 5. Chương trình đào tạo\nCT CLC được xây ...,Văn bản: QUY ĐỊNH ĐÀO TẠO CHƯƠNG TRÌNH CHẤT LƯ...,479,37,content_2b671f3959191064a689


In [6]:
# ============================================================
# 5. LOAD AN EVALUATION SPLIT
# ============================================================

def load_eval_split(split_name: str):
    query_path = PROCESSED_DIR / f"{split_name}_queries.json"
    qrels_path = PROCESSED_DIR / f"{split_name}_qrels.json"

    if not query_path.exists() or not qrels_path.exists():
        raise FileNotFoundError(
            f"Không tìm thấy queries/qrels cho split '{split_name}'."
        )

    with query_path.open("r", encoding="utf-8") as file:
        queries = json.load(file)

    with qrels_path.open("r", encoding="utf-8") as file:
        qrels = json.load(file)

    valid_query_ids = []
    missing_positive_ids = set()

    for query_id, relevant_ids in qrels.items():
        missing = [
            passage_id
            for passage_id in relevant_ids
            if passage_id not in passage_id_to_index
        ]

        if missing:
            missing_positive_ids.update(missing)
            continue

        if query_id in queries:
            valid_query_ids.append(query_id)

    if missing_positive_ids:
        print(
            f"Cảnh báo: bỏ {len(missing_positive_ids)} passage ID "
            "không có trong corpus."
        )

    query_texts = [queries[query_id] for query_id in valid_query_ids]
    filtered_qrels = {
        query_id: qrels[query_id]
        for query_id in valid_query_ids
    }

    return valid_query_ids, query_texts, filtered_qrels


for split_name in EVAL_SPLITS:
    query_ids, query_texts, qrels = load_eval_split(split_name)
    print(
        f"{split_name:>12}: "
        f"{len(query_ids):,} queries | "
        f"{sum(len(v) for v in qrels.values()):,} relevance labels"
    )

         val: 971 queries | 971 relevance labels
  strict_val: 933 queries | 933 relevance labels


In [7]:
# ============================================================
# 6. MODEL-SPECIFIC TEXT FORMATTING
# ============================================================

def qwen_query_instruction(question: str) -> str:
    return (
        f"Instruct: {QWEN_TASK_INSTRUCTION}\n"
        f"Query:{question}"
    )


def format_texts(
    texts: list[str],
    format_name: str,
    word_segment: bool = False,
) -> list[str]:
    if format_name == "qwen_instruction":
        formatted = [qwen_query_instruction(text) for text in texts]
    elif format_name == "e5_query":
        formatted = [f"query: {text}" for text in texts]
    elif format_name == "e5_passage":
        formatted = [f"passage: {text}" for text in texts]
    elif format_name == "raw":
        formatted = list(texts)
    else:
        raise ValueError(f"Unknown text format: {format_name}")

    if word_segment:
        # BKAI/PhoBERT yêu cầu word-segmented Vietnamese input.
        formatted = [
            word_tokenize(text, format="text")
            for text in formatted
        ]

    return formatted


sample_question = "Sinh viên được bảo lưu kết quả học tập trong trường hợp nào?"

for model_key, spec in MODEL_SPECS.items():
    example = format_texts(
        [sample_question],
        spec.query_format,
        spec.word_segment_vietnamese,
    )[0]

    print(f"\n[{model_key}]\n{example[:500]}")


[bge_m3]
Sinh viên được bảo lưu kết quả học tập trong trường hợp nào?

[multilingual_e5_base]
query: Sinh viên được bảo lưu kết quả học tập trong trường hợp nào?

[bkai_vietnamese_bi_encoder]
Sinh_viên được bảo_lưu kết_quả học_tập trong trường_hợp nào ?


In [8]:
# ============================================================
# 7. MODEL LOADING AND CLEANUP
# ============================================================

def clear_gpu_memory() -> None:
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.reset_peak_memory_stats()


def load_sentence_transformer(spec: ModelSpec) -> SentenceTransformer:
    clear_gpu_memory()

    common_kwargs = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "model_kwargs": {
            "torch_dtype": TORCH_DTYPE,
        },
    }

    if spec.key == "qwen3_embedding_4b":
        common_kwargs["tokenizer_kwargs"] = {
            "padding_side": "left",
        }

        # FlashAttention có thể chưa được cài trên mọi runtime.
        # Thử flash_attention_2 trước, sau đó tự fallback sang SDPA.
        if spec.use_flash_attention:
            try:
                model = SentenceTransformer(
                    spec.model_name,
                    model_kwargs={
                        "torch_dtype": TORCH_DTYPE,
                        "attn_implementation": "flash_attention_2",
                    },
                    tokenizer_kwargs={"padding_side": "left"},
                    device=common_kwargs["device"],
                    truncate_dim=spec.truncate_dim,
                )
                print("Qwen loaded with FlashAttention 2.")
                return model
            except Exception as error:
                print(
                    "Không dùng được FlashAttention 2, fallback sang SDPA."
                )
                print("Reason:", type(error).__name__, str(error)[:300])

                clear_gpu_memory()

                return SentenceTransformer(
                    spec.model_name,
                    model_kwargs={
                        "torch_dtype": TORCH_DTYPE,
                        "attn_implementation": "sdpa",
                    },
                    tokenizer_kwargs={"padding_side": "left"},
                    device=common_kwargs["device"],
                    truncate_dim=spec.truncate_dim,
                )

    return SentenceTransformer(
        spec.model_name,
        **common_kwargs,
    )


def unload_model(model: SentenceTransformer | None) -> None:
    if model is not None:
        try:
            model.to("cpu")
        except Exception:
            pass

        del model

    clear_gpu_memory()

In [9]:
# ============================================================
# 8. EMBEDDING CACHE
# ============================================================

def safe_cache_name(value: str) -> str:
    return (
        value.replace("/", "__")
        .replace(" ", "_")
        .replace(":", "_")
    )


def embedding_cache_path(
    model_key: str,
    data_key: str,
    max_length: int,
    corpus_field: str | None = None,
) -> Path:
    parts = [
        safe_cache_name(model_key),
        safe_cache_name(data_key),
        f"maxlen_{max_length}",
    ]

    if corpus_field:
        parts.append(safe_cache_name(corpus_field))

    return EMBEDDING_CACHE_DIR / ("__".join(parts) + ".npy")


def save_embedding_cache(
    embeddings: np.ndarray,
    path: Path,
) -> None:
    temporary_path = path.with_suffix(".tmp.npy")
    np.save(temporary_path, embeddings.astype(np.float32))
    temporary_path.replace(path)


def load_embedding_cache(path: Path) -> np.ndarray | None:
    if (
        USE_EMBEDDING_CACHE
        and not FORCE_RECOMPUTE
        and path.exists()
    ):
        print("Load cache:", path.name)
        return np.load(path)

    return None

In [10]:
# ============================================================
# 9. ENCODE FUNCTION WITH TIME AND VRAM MEASUREMENT
# ============================================================

@dataclass
class EncodeStats:
    num_texts: int
    seconds: float
    texts_per_second: float
    peak_vram_gb: float
    embedding_dim: int
    cache_hit: bool


def encode_texts(
    model: SentenceTransformer,
    texts: list[str],
    batch_size: int,
    max_length: int,
    cache_path: Path,
    show_progress_bar: bool = True,
) -> tuple[np.ndarray, EncodeStats]:

    cached_embeddings = load_embedding_cache(cache_path)

    if cached_embeddings is not None:
        return cached_embeddings, EncodeStats(
            num_texts=len(cached_embeddings),
            seconds=0.0,
            texts_per_second=float("inf"),
            peak_vram_gb=0.0,
            embedding_dim=int(cached_embeddings.shape[1]),
            cache_hit=True,
        )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    model.max_seq_length = max_length

    start_time = time.perf_counter()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start_time

    peak_vram_gb = (
        torch.cuda.max_memory_allocated() / 1024**3
        if torch.cuda.is_available()
        else 0.0
    )

    embeddings = np.asarray(embeddings, dtype=np.float32)

    if USE_EMBEDDING_CACHE:
        save_embedding_cache(embeddings, cache_path)

    stats = EncodeStats(
        num_texts=len(texts),
        seconds=elapsed,
        texts_per_second=(
            len(texts) / elapsed if elapsed > 0 else float("inf")
        ),
        peak_vram_gb=peak_vram_gb,
        embedding_dim=int(embeddings.shape[1]),
        cache_hit=False,
    )

    return embeddings, stats

In [11]:
# ============================================================
# 10. RETRIEVAL AND METRICS
# ============================================================

def retrieve_top_k(
    query_embeddings: np.ndarray,
    corpus_embeddings: np.ndarray,
    top_k: int,
    query_chunk_size: int = 256,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Cosine similarity vì embeddings đã được L2-normalized.
    Tính theo chunk query để tránh tạo matrix quá lớn.
    """
    top_k = min(top_k, corpus_embeddings.shape[0])

    all_indices = []
    all_scores = []

    corpus_tensor = torch.from_numpy(corpus_embeddings).to(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    for start in range(0, len(query_embeddings), query_chunk_size):
        end = min(start + query_chunk_size, len(query_embeddings))

        query_tensor = torch.from_numpy(
            query_embeddings[start:end]
        ).to(corpus_tensor.device)

        scores = query_tensor @ corpus_tensor.T

        chunk_scores, chunk_indices = torch.topk(
            scores,
            k=top_k,
            dim=1,
            largest=True,
            sorted=True,
        )

        all_scores.append(chunk_scores.float().cpu().numpy())
        all_indices.append(chunk_indices.cpu().numpy())

        del query_tensor, scores, chunk_scores, chunk_indices

    del corpus_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        np.concatenate(all_indices, axis=0),
        np.concatenate(all_scores, axis=0),
    )


def dcg_at_k(relevance: list[int], k: int) -> float:
    score = 0.0

    for rank, rel in enumerate(relevance[:k], start=1):
        if rel:
            score += rel / math.log2(rank + 1)

    return score


def evaluate_rankings(
    query_ids: list[str],
    ranked_indices: np.ndarray,
    qrels: dict[str, list[str]],
    k_values: list[int],
) -> dict[str, float]:
    recall_totals = {k: 0.0 for k in k_values}
    reciprocal_ranks = []
    ndcg_scores = []
    average_precisions = []

    for row_index, query_id in enumerate(query_ids):
        relevant_ids = set(qrels[query_id])
        relevant_count = len(relevant_ids)

        ranked_passage_ids = [
            passage_ids[index]
            for index in ranked_indices[row_index]
        ]

        relevance = [
            1 if passage_id in relevant_ids else 0
            for passage_id in ranked_passage_ids
        ]

        for k in k_values:
            retrieved_relevant = sum(relevance[:k])
            recall_totals[k] += (
                retrieved_relevant / relevant_count
                if relevant_count > 0
                else 0.0
            )

        first_relevant_rank = next(
            (
                rank
                for rank, rel in enumerate(relevance[:10], start=1)
                if rel
            ),
            None,
        )

        reciprocal_ranks.append(
            1.0 / first_relevant_rank
            if first_relevant_rank is not None
            else 0.0
        )

        ideal_relevance = [1] * min(relevant_count, 10)
        ideal_dcg = dcg_at_k(ideal_relevance, 10)
        actual_dcg = dcg_at_k(relevance, 10)

        ndcg_scores.append(
            actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0
        )

        hits = 0
        precision_sum = 0.0

        for rank, rel in enumerate(relevance[:10], start=1):
            if rel:
                hits += 1
                precision_sum += hits / rank

        average_precisions.append(
            precision_sum / min(relevant_count, 10)
            if relevant_count > 0
            else 0.0
        )

    num_queries = len(query_ids)

    metrics = {
        f"Recall@{k}": recall_totals[k] / num_queries
        for k in k_values
    }

    metrics.update(
        {
            "MRR@10": float(np.mean(reciprocal_ranks)),
            "nDCG@10": float(np.mean(ndcg_scores)),
            "MAP@10": float(np.mean(average_precisions)),
        }
    )

    return metrics

In [12]:
# ============================================================
# 11. SAVE RETRIEVAL EXAMPLES FOR ERROR ANALYSIS
# ============================================================

def build_retrieval_examples(
    split_name: str,
    model_key: str,
    corpus_field: str,
    query_ids: list[str],
    query_texts: list[str],
    ranked_indices: np.ndarray,
    ranked_scores: np.ndarray,
    qrels: dict[str, list[str]],
    num_queries: int = 50,
    top_k: int = 10,
) -> pd.DataFrame:
    rows = []

    sample_size = min(num_queries, len(query_ids))
    sampled_positions = np.linspace(
        0,
        len(query_ids) - 1,
        sample_size,
        dtype=int,
    )

    for position in sampled_positions:
        query_id = query_ids[position]
        relevant_ids = set(qrels[query_id])

        for rank in range(min(top_k, ranked_indices.shape[1])):
            corpus_index = int(ranked_indices[position, rank])
            passage = corpus_df.iloc[corpus_index]

            rows.append(
                {
                    "split": split_name,
                    "model_key": model_key,
                    "corpus_field": corpus_field,
                    "query_id": query_id,
                    "question": query_texts[position],
                    "rank": rank + 1,
                    "score": float(ranked_scores[position, rank]),
                    "is_relevant": passage["passage_id"] in relevant_ids,
                    "passage_id": passage["passage_id"],
                    "document": passage["document"],
                    "article": passage["article"],
                    "content": passage["content"],
                }
            )

    return pd.DataFrame(rows)

In [13]:
# ============================================================
# 12. BENCHMARK ONE MODEL
# ============================================================

def benchmark_model(
    spec: ModelSpec,
    eval_splits: list[str],
    corpus_fields: list[str],
) -> tuple[list[dict], list[pd.DataFrame]]:
    results = []
    retrieval_examples = []
    model = None

    try:
        model = load_sentence_transformer(spec)

        model_card = {
            "model_key": spec.key,
            "model_name": spec.model_name,
            "max_query_length": spec.max_query_length,
            "max_passage_length": spec.max_passage_length,
            "query_format": spec.query_format,
            "passage_format": spec.passage_format,
            "word_segment_vietnamese": spec.word_segment_vietnamese,
            "truncate_dim": spec.truncate_dim,
        }

        with (
            OUTPUT_DIR / f"{spec.key}_benchmark_config.json"
        ).open("w", encoding="utf-8") as file:
            json.dump(
                model_card,
                file,
                ensure_ascii=False,
                indent=2,
            )

        for corpus_field in corpus_fields:
            raw_passages = (
                corpus_df[corpus_field]
                .fillna("")
                .astype(str)
                .tolist()
            )

            formatted_passages = format_texts(
                raw_passages,
                spec.passage_format,
                spec.word_segment_vietnamese,
            )

            corpus_cache_path = embedding_cache_path(
                model_key=spec.key,
                data_key="corpus",
                max_length=spec.max_passage_length,
                corpus_field=corpus_field,
            )

            corpus_embeddings, corpus_stats = encode_texts(
                model=model,
                texts=formatted_passages,
                batch_size=spec.passage_batch_size,
                max_length=spec.max_passage_length,
                cache_path=corpus_cache_path,
            )

            for split_name in eval_splits:
                query_ids, raw_queries, qrels = load_eval_split(
                    split_name
                )

                formatted_queries = format_texts(
                    raw_queries,
                    spec.query_format,
                    spec.word_segment_vietnamese,
                )

                query_cache_path = embedding_cache_path(
                    model_key=spec.key,
                    data_key=f"queries_{split_name}",
                    max_length=spec.max_query_length,
                    corpus_field=None,
                )

                query_embeddings, query_stats = encode_texts(
                    model=model,
                    texts=formatted_queries,
                    batch_size=spec.query_batch_size,
                    max_length=spec.max_query_length,
                    cache_path=query_cache_path,
                )

                retrieval_start = time.perf_counter()

                ranked_indices, ranked_scores = retrieve_top_k(
                    query_embeddings=query_embeddings,
                    corpus_embeddings=corpus_embeddings,
                    top_k=MAX_RANK,
                )

                retrieval_seconds = (
                    time.perf_counter() - retrieval_start
                )

                metrics = evaluate_rankings(
                    query_ids=query_ids,
                    ranked_indices=ranked_indices,
                    qrels=qrels,
                    k_values=TOP_K_VALUES,
                )

                result = {
                    "model_key": spec.key,
                    "model_name": spec.model_name,
                    "split": split_name,
                    "corpus_field": corpus_field,
                    "num_queries": len(query_ids),
                    "num_passages": len(corpus_df),
                    "embedding_dim": corpus_stats.embedding_dim,
                    "query_max_length": spec.max_query_length,
                    "passage_max_length": spec.max_passage_length,
                    "corpus_encode_seconds": corpus_stats.seconds,
                    "corpus_texts_per_second": (
                        corpus_stats.texts_per_second
                    ),
                    "query_encode_seconds": query_stats.seconds,
                    "query_texts_per_second": (
                        query_stats.texts_per_second
                    ),
                    "retrieval_seconds": retrieval_seconds,
                    "corpus_peak_vram_gb": (
                        corpus_stats.peak_vram_gb
                    ),
                    "query_peak_vram_gb": (
                        query_stats.peak_vram_gb
                    ),
                    "corpus_cache_hit": corpus_stats.cache_hit,
                    "query_cache_hit": query_stats.cache_hit,
                    **metrics,
                }

                results.append(result)

                examples_df = build_retrieval_examples(
                    split_name=split_name,
                    model_key=spec.key,
                    corpus_field=corpus_field,
                    query_ids=query_ids,
                    query_texts=raw_queries,
                    ranked_indices=ranked_indices,
                    ranked_scores=ranked_scores,
                    qrels=qrels,
                )

                retrieval_examples.append(examples_df)

                print(
                    f"{spec.key} | {split_name} | "
                    f"{corpus_field} | "
                    f"R@1={metrics['Recall@1']:.4f} | "
                    f"R@10={metrics['Recall@10']:.4f} | "
                    f"MRR@10={metrics['MRR@10']:.4f}"
                )

                del (
                    query_embeddings,
                    ranked_indices,
                    ranked_scores,
                )
                clear_gpu_memory()

            del corpus_embeddings
            clear_gpu_memory()

    finally:
        unload_model(model)

    return results, retrieval_examples

In [14]:
# ============================================================
# 13. RUN VALIDATION BENCHMARK
# ============================================================

all_results = []
all_examples = []

for model_key, spec in MODEL_SPECS.items():
    print("\n" + "=" * 100)
    print("BENCHMARK:", model_key)
    print("=" * 100)

    try:
        model_results, model_examples = benchmark_model(
            spec=spec,
            eval_splits=EVAL_SPLITS,
            corpus_fields=CORPUS_FIELDS,
        )

        all_results.extend(model_results)
        all_examples.extend(model_examples)

        partial_results_df = pd.DataFrame(all_results)

        partial_results_df.to_csv(
            OUTPUT_DIR / "validation_benchmark_results.csv",
            index=False,
            encoding="utf-8-sig",
        )

        partial_results_df.to_parquet(
            OUTPUT_DIR / "validation_benchmark_results.parquet",
            index=False,
        )

    except Exception as error:
        print(
            f"Model {model_key} failed: "
            f"{type(error).__name__}: {error}"
        )

        error_record = {
            "model_key": model_key,
            "error_type": type(error).__name__,
            "error_message": str(error),
        }

        with (
            OUTPUT_DIR / f"{model_key}_error.json"
        ).open("w", encoding="utf-8") as file:
            json.dump(
                error_record,
                file,
                ensure_ascii=False,
                indent=2,
            )

        clear_gpu_memory()

benchmark_df = pd.DataFrame(all_results)

if benchmark_df.empty:
    raise RuntimeError(
        "Không model nào benchmark thành công. "
        "Kiểm tra các file *_error.json."
    )

benchmark_df.to_csv(
    OUTPUT_DIR / "validation_benchmark_results.csv",
    index=False,
    encoding="utf-8-sig",
)

benchmark_df.to_parquet(
    OUTPUT_DIR / "validation_benchmark_results.parquet",
    index=False,
)

if all_examples:
    examples_df = pd.concat(all_examples, ignore_index=True)
    examples_df.to_parquet(
        OUTPUT_DIR / "retrieval_examples.parquet",
        index=False,
    )

display(
    benchmark_df.sort_values(
        by=["split", "Recall@10", "MRR@10"],
        ascending=[True, False, False],
    )
)


BENCHMARK: bge_m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Load cache: bkai_vietnamese_bi_encoder__corpus__maxlen_256__content.npy
Load cache: bkai_vietnamese_bi_encoder__queries_val__maxlen_128.npy
bkai_vietnamese_bi_encoder | val | content | R@1=0.1576 | R@10=0.6550 | MRR@10=0.3043
Load cache: bkai_vietnamese_bi_encoder__queries_strict_val__maxlen_128.npy
bkai_vietnamese_bi_encoder | strict_val | content | R@1=0.1533 | R@10=0.6506 | MRR@10=0.3024
Load cache: bkai_vietnamese_bi_encoder__corpus__maxlen_256__embedding_text.npy
Load cache: bkai_vietnamese_bi_encoder__queries_val__maxlen_128.npy
bkai_vietnamese_bi_encoder | val | embedding_text | R@1=0.1802 | R@10=0.6365 | MRR@10=0.3165
Load cache: bkai_vietnamese_bi_encoder__queries_strict_val__maxlen_128.npy
bkai_vietnamese_bi_encoder | strict_val | embedding_text | R@1=0.1801 | R@10=0.6377 | MRR@10=0.3168


,model_key,model_name,split,corpus_field,num_queries,num_passages,embedding_dim,query_max_length,passage_max_length,corpus_encode_seconds,...,corpus_cache_hit,query_cache_hit,Recall@1,Recall@3,Recall@5,Recall@10,Recall@20,MRR@10,nDCG@10,MAP@10
3,bge_m3,BAAI/bge-m3,strict_val,embedding_text,933,1045,1024,128,512,0.0,...,True,True,0.224009,0.472669,0.618435,0.752412,0.823151,0.385654,0.473461,0.385654
1,bge_m3,BAAI/bge-m3,strict_val,content,933,1045,1024,128,512,0.0,...,True,True,0.196141,0.480171,0.616292,0.736334,0.844587,0.365397,0.454705,0.365397
5,multilingual_e5_base,intfloat/multilingual-e5-base,strict_val,content,933,1045,768,128,512,0.0,...,True,True,0.182208,0.430868,0.577706,0.732047,0.843516,0.344139,0.436660,0.344139
7,multilingual_e5_base,intfloat/multilingual-e5-base,strict_val,embedding_text,933,1045,768,128,512,0.0,...,True,True,0.228296,0.470525,0.579850,0.728832,0.825295,0.378474,0.462030,0.378474
9,bkai_vietnamese_bi_encoder,bkai-foundation-models/vietnamese-bi-encoder,strict_val,content,933,1045,768,128,256,0.0,...,True,True,0.153269,0.398714,0.521972,0.650589,0.751340,0.302450,0.385757,0.302450
11,bkai_vietnamese_bi_encoder,bkai-foundation-models/vietnamese-bi-encoder,strict_val,embedding_text,933,1045,768,128,256,0.0,...,True,True,0.180064,0.394427,0.511254,0.637728,0.724544,0.316814,0.393343,0.316814
2,bge_m3,BAAI/bge-m3,val,embedding_text,971,1045,1024,128,512,0.0,...,True,True,0.222451,0.471679,0.617920,0.750772,0.820803,0.384301,0.472079,0.384301
0,bge_m3,BAAI/bge-m3,val,content,971,1045,1024,128,512,0.0,...,True,True,0.196704,0.474768,0.615860,0.735324,0.843460,0.365202,0.454291,0.365202
4,multilingual_e5_base,intfloat/multilingual-e5-base,val,content,971,1045,768,128,512,0.0,...,True,True,0.182286,0.427394,0.575695,0.731205,0.841401,0.343327,0.435801,0.343327
6,multilingual_e5_base,intfloat/multilingual-e5-base,val,embedding_text,971,1045,768,128,512,0.0,...,True,True,0.224511,0.466529,0.578785,0.727085,0.823893,0.375321,0.459233,0.375321


In [15]:
# ============================================================
# 14. MODEL SELECTION TABLE
# ============================================================

# Ưu tiên strict_val nếu có, nếu không dùng val.
selection_split = (
    "strict_val"
    if "strict_val" in benchmark_df["split"].unique()
    else "val"
)

selection_df = benchmark_df.loc[
    benchmark_df["split"].eq(selection_split)
].copy()

# Composite score chỉ dùng để hỗ trợ quan sát.
# Quyết định chính vẫn nên dựa trên Recall@k, MRR và latency.
selection_df["quality_score"] = (
    0.40 * selection_df["Recall@5"]
    + 0.30 * selection_df["Recall@10"]
    + 0.20 * selection_df["MRR@10"]
    + 0.10 * selection_df["nDCG@10"]
)

selection_df = selection_df.sort_values(
    by=[
        "quality_score",
        "Recall@10",
        "MRR@10",
        "query_texts_per_second",
    ],
    ascending=[False, False, False, False],
)

selection_df.to_csv(
    OUTPUT_DIR / "model_selection_table.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    selection_df[
        [
            "model_key",
            "model_name",
            "corpus_field",
            "Recall@1",
            "Recall@5",
            "Recall@10",
            "MRR@10",
            "nDCG@10",
            "quality_score",
            "query_texts_per_second",
            "corpus_texts_per_second",
            "embedding_dim",
            "corpus_peak_vram_gb",
        ]
    ]
)

print("\nRecommended by validation:")
print(selection_df.iloc[0][
    ["model_key", "model_name", "corpus_field", "quality_score"]
])

,model_key,model_name,corpus_field,Recall@1,Recall@5,Recall@10,MRR@10,nDCG@10,quality_score,query_texts_per_second,corpus_texts_per_second,embedding_dim,corpus_peak_vram_gb
3,bge_m3,BAAI/bge-m3,embedding_text,0.224009,0.618435,0.752412,0.385654,0.473461,0.597575,inf,inf,1024,0.0
1,bge_m3,BAAI/bge-m3,content,0.196141,0.616292,0.736334,0.365397,0.454705,0.585967,inf,inf,1024,0.0
7,multilingual_e5_base,intfloat/multilingual-e5-base,embedding_text,0.228296,0.579850,0.728832,0.378474,0.462030,0.572487,inf,inf,768,0.0
5,multilingual_e5_base,intfloat/multilingual-e5-base,content,0.182208,0.577706,0.732047,0.344139,0.436660,0.563191,inf,inf,768,0.0
9,bkai_vietnamese_bi_encoder,bkai-foundation-models/vietnamese-bi-encoder,content,0.153269,0.521972,0.650589,0.302450,0.385757,0.503031,inf,inf,768,0.0
11,bkai_vietnamese_bi_encoder,bkai-foundation-models/vietnamese-bi-encoder,embedding_text,0.180064,0.511254,0.637728,0.316814,0.393343,0.498517,inf,inf,768,0.0



Recommended by validation:
model_key                bge_m3
model_name          BAAI/bge-m3
corpus_field     embedding_text
quality_score          0.597575
Name: 3, dtype: object


In [16]:
# ============================================================
# 15. OPTIONAL: FINAL TEST FOR THE SELECTED MODEL ONLY
# ============================================================

if RUN_FINAL_TEST:
    if SELECTED_MODEL_KEY not in MODEL_SPECS:
        raise KeyError(
            f"Unknown SELECTED_MODEL_KEY: {SELECTED_MODEL_KEY}"
        )

    selected_spec = MODEL_SPECS[SELECTED_MODEL_KEY]

    selected_validation_rows = selection_df.loc[
        selection_df["model_key"].eq(SELECTED_MODEL_KEY)
    ]

    if selected_validation_rows.empty:
        raise ValueError(
            f"{SELECTED_MODEL_KEY} chưa có kết quả validation."
        )

    selected_corpus_field = (
        selected_validation_rows.iloc[0]["corpus_field"]
    )

    print("Final model:", SELECTED_MODEL_KEY)
    print("Corpus field:", selected_corpus_field)

    test_results, test_examples = benchmark_model(
        spec=selected_spec,
        eval_splits=FINAL_TEST_SPLITS,
        corpus_fields=[selected_corpus_field],
    )

    test_results_df = pd.DataFrame(test_results)

    test_results_df.to_csv(
        OUTPUT_DIR / "final_test_results.csv",
        index=False,
        encoding="utf-8-sig",
    )

    test_results_df.to_parquet(
        OUTPUT_DIR / "final_test_results.parquet",
        index=False,
    )

    if test_examples:
        pd.concat(
            test_examples,
            ignore_index=True,
        ).to_parquet(
            OUTPUT_DIR / "final_test_retrieval_examples.parquet",
            index=False,
        )

    display(test_results_df)
else:
    print(
        "RUN_FINAL_TEST=False: chưa sử dụng test để tránh model selection leakage."
    )

RUN_FINAL_TEST=False: chưa sử dụng test để tránh model selection leakage.


In [17]:
# ============================================================
# 16. EXPORT WINNER CONFIG FOR THE NEXT NOTEBOOK
# ============================================================

best_row = selection_df.iloc[0]

winner_config = {
    "selected_on_split": selection_split,
    "model_key": best_row["model_key"],
    "model_name": best_row["model_name"],
    "corpus_field": best_row["corpus_field"],
    "query_max_length": int(best_row["query_max_length"]),
    "passage_max_length": int(best_row["passage_max_length"]),
    "embedding_dim": int(best_row["embedding_dim"]),
    "metrics": {
        metric: float(best_row[metric])
        for metric in [
            "Recall@1",
            "Recall@3",
            "Recall@5",
            "Recall@10",
            "Recall@20",
            "MRR@10",
            "nDCG@10",
            "MAP@10",
        ]
    },
    "qwen_task_instruction": (
        QWEN_TASK_INSTRUCTION
        if best_row["model_key"] == "qwen3_embedding_4b"
        else None
    ),
    "note": (
        "This file records the zero-shot benchmark winner. "
        "The next notebook may still mine hard negatives and compare "
        "fine-tuned checkpoints against this baseline."
    ),
}

with (
    OUTPUT_DIR / "zero_shot_winner_config.json"
).open("w", encoding="utf-8") as file:
    json.dump(
        winner_config,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    json.dumps(
        winner_config,
        ensure_ascii=False,
        indent=2,
    )
)

{
  "selected_on_split": "strict_val",
  "model_key": "bge_m3",
  "model_name": "BAAI/bge-m3",
  "corpus_field": "embedding_text",
  "query_max_length": 128,
  "passage_max_length": 512,
  "embedding_dim": 1024,
  "metrics": {
    "Recall@1": 0.2240085744908896,
    "Recall@3": 0.47266881028938906,
    "Recall@5": 0.6184351554126474,
    "Recall@10": 0.752411575562701,
    "Recall@20": 0.8231511254019293,
    "MRR@10": 0.38565431531669475,
    "nDCG@10": 0.4734614605363139,
    "MAP@10": 0.38565431531669475
  },
  "qwen_task_instruction": null,
  "note": "This file records the zero-shot benchmark winner. The next notebook may still mine hard negatives and compare fine-tuned checkpoints against this baseline."
}


In [18]:
# ============================================================
# 17. FINAL OUTPUT CHECK
# ============================================================

required_outputs = [
    OUTPUT_DIR / "validation_benchmark_results.csv",
    OUTPUT_DIR / "validation_benchmark_results.parquet",
    OUTPUT_DIR / "model_selection_table.csv",
    OUTPUT_DIR / "zero_shot_winner_config.json",
]

missing_outputs = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise RuntimeError(
        "Thiếu output:\n- " + "\n- ".join(missing_outputs)
    )

print("Benchmark hoàn tất.\n")

for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file():
        print(
            f"{path.name:48s} "
            f"{path.stat().st_size / 1024:12.2f} KB"
        )

Benchmark hoàn tất.

bge_m3_benchmark_config.json                             0.22 KB
bkai_vietnamese_bi_encoder_benchmark_config.json         0.27 KB
model_selection_table.csv                                2.24 KB
multilingual_e5_base_benchmark_config.json               0.26 KB
qwen3_embedding_0_6b_benchmark_config.json               0.26 KB
qwen3_embedding_4b_benchmark_config.json                 0.26 KB
retrieval_examples.parquet                             244.39 KB
validation_benchmark_results.csv                         3.82 KB
validation_benchmark_results.parquet                    16.75 KB
zero_shot_winner_config.json                             0.70 KB


# Đầu ra của notebook

```text
artifacts/embedding_benchmark/
├── embedding_cache/
│   └── *.npy
├── validation_benchmark_results.csv
├── validation_benchmark_results.parquet
├── model_selection_table.csv
├── retrieval_examples.parquet
├── zero_shot_winner_config.json
├── <model>_benchmark_config.json
└── final_test_results.*              # chỉ khi RUN_FINAL_TEST=True
```

## Bước tiếp theo

Notebook tiếp theo không fine-tune ngay. Nó sẽ dùng kết quả zero-shot để:

1. build BM25;
2. lấy các passage sai nhưng được xếp hạng cao;
3. mine hard negatives từ BM25 và Dense Retriever;
4. loại false negatives theo `passage_id`, document/article và answer overlap;
5. tạo triplet hoặc group training data cho Qwen3 Embedding.